<a href="https://colab.research.google.com/github/armandoordonez/eda_couse/blob/main/Pipelines_ColumnTransformer_Guia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipelines y ColumnTransformer en scikit-learn — Guía paso a paso


1. **El problema** — Haremos el preprocesamiento "a mano", paso por paso, y descubriremos por qué esta forma de trabajar es frágil, repetitiva y —lo más grave— **puede arruinar silenciosamente un modelo** por una fuga de información (*data leakage*).

2. **La solución** — Introduciremos el `Pipeline` y el `ColumnTransformer` para resolver esos problemas, entendiendo cada pieza antes de ensamblarla.

3. **El potencial** — Veremos lo que un pipeline bien construido nos regala "gratis": validación cruzada correcta, búsqueda de hiperparámetros, y un modelo listo para producción en una sola línea.

> **Idea central que debe quedar clara al terminar:** un pipeline no es un simple atajo de código. Es la garantía de que **exactamente las mismas transformaciones** se aplican a los datos de entrenamiento, de prueba y de producción, aprendiendo sus parámetros **sólo** de los datos de entrenamiento.

---
# Acto 1 — El problema: preprocesamiento "a mano"

Antes de valorar una solución, hay que sentir el problema que resuelve. En este primer acto haremos las cosas de la manera "intuitiva" —transformando los datos paso a paso— y expondremos sus tres debilidades.

## 1.1. Librerías y carga de datos

In [1]:
# Manipulación de datos
import pandas as pd
import numpy as np

# Herramientas de scikit-learn que iremos presentando a lo largo del cuaderno
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

pd.set_option("display.max_columns", None)
print("Librerías importadas correctamente.")

Librerías importadas correctamente.


In [2]:
# Cargamos el dataset de viviendas de California.
url = "https://raw.githubusercontent.com/armandoordonez/eda_couse/main/data/housing.csv"
housing = pd.read_csv(url)

print("Dimensiones:", housing.shape)
housing.head()

Dimensiones: (20640, 10)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## 1.2. Un vistazo rápido al conjunto de datos

Nuestro objetivo es **predecir `median_house_value`** (el valor mediano de la vivienda) a partir del resto de columnas. Observemos dos características que serán decisivas para el resto del cuaderno:

- La mayoría de columnas son **numéricas** (longitud, latitud, número de habitaciones, ingreso mediano, etc.).
- Hay **una columna categórica**: `ocean_proximity` (cercanía al océano), con texto como `NEAR BAY`, `INLAND`, etc.
- La columna `total_bedrooms` tiene **valores perdidos**.

Esta mezcla de tipos —numéricas y categóricas en el mismo dataset— es exactamente el escenario donde el `ColumnTransformer` brilla, porque **cada tipo necesita un tratamiento diferente**.

In [3]:
# Tipos de datos y valores perdidos
print("Tipos de datos:")
print(housing.dtypes)
print("\nValores perdidos por columna:")
print(housing.isna().sum()[housing.isna().sum() > 0])
print("\nCategorías de ocean_proximity:")
print(housing["ocean_proximity"].value_counts())

Tipos de datos:
longitude             float64
latitude              float64
housing_median_age    float64
total_rooms           float64
total_bedrooms        float64
population            float64
households            float64
median_income         float64
median_house_value    float64
ocean_proximity        object
dtype: object

Valores perdidos por columna:
total_bedrooms    207
dtype: int64

Categorías de ocean_proximity:
ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64


## 1.3. Separación en entrenamiento y prueba

**Este es el paso más importante para entender el resto del cuaderno.** Separamos los datos *antes* de transformar cualquier cosa.

¿Por qué? Porque el conjunto de prueba debe simular **datos que el modelo nunca ha visto**. Si calculamos, por ejemplo, la media para rellenar faltantes usando *todos* los datos (incluido el test), estaríamos filtrando información del futuro hacia el entrenamiento. A esto se le llama **fuga de información (*data leakage*)** y es uno de los errores más costosos y difíciles de detectar en la práctica.

In [4]:
# X = variables predictoras ; y = variable objetivo
X = housing.drop("median_house_value", axis=1)
y = housing["median_house_value"]

# Separamos ANTES de transformar. El test queda "en una caja sellada".
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Entrenamiento:", X_train.shape)
print("Prueba:       ", X_test.shape)

Entrenamiento: (16512, 9)
Prueba:        (4128, 9)


## 1.4. Preprocesamiento manual  

Sin usar pipelines:

1. Rellenar los faltantes de `total_bedrooms` con la **mediana**.
2. **Escalar** las variables numéricas (para que estén en una escala comparable).
3. Aplicar **One-Hot Encoding** a la variable categórica `ocean_proximity`.

Preste mucha atención a cuántos pasos manuales y cuánta disciplina exige hacer esto **correctamente**.

In [16]:
# --- Trabajaremos sobre copias para no dañar los datos originales ---
X_train_manual = X_train.copy()
X_test_manual = X_test.copy()

numericas = ["longitude", "latitude", "housing_median_age", "total_rooms",
             "total_bedrooms", "population", "households", "median_income"]
categoricas = ["ocean_proximity"]

# --- PASO 1: Imputar faltantes ---
# CLAVE: la mediana se calcula SÓLO con el entrenamiento...
mediana_bedrooms = X_train_manual["total_bedrooms"].median()
# ...y ese MISMO valor se usa para rellenar tanto train como test.
X_train_manual["total_bedrooms"] = X_train_manual["total_bedrooms"].fillna(mediana_bedrooms)
X_test_manual["total_bedrooms"]  = X_test_manual["total_bedrooms"].fillna(mediana_bedrooms)

# --- PASO 2: Escalar numéricas ---
# CLAVE: el scaler APRENDE (fit) media y desviación SÓLO del train...
scaler = StandardScaler()
X_train_manual[numericas] = scaler.fit_transform(X_train_manual[numericas])
# ...y sólo TRANSFORMA el test con esos parámetros aprendidos.
X_test_manual[numericas] = scaler.transform(X_test_manual[numericas])

# --- PASO 3: One-Hot Encoding de la categórica ---
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
train_ohe = ohe.fit_transform(X_train_manual[categoricas])
test_ohe = ohe.transform(X_test_manual[categoricas])

# Reconstruir los DataFrames con las columnas codificadas (paso tedioso y propenso a errores)
cols_ohe = ohe.get_feature_names_out(categoricas)
train_ohe_df = pd.DataFrame(train_ohe, columns=cols_ohe, index=X_train_manual.index)
test_ohe_df = pd.DataFrame(test_ohe, columns=cols_ohe, index=X_test_manual.index)

X_train_manual = pd.concat([X_train_manual[numericas], train_ohe_df], axis=1)
X_test_manual = pd.concat([X_test_manual[numericas], test_ohe_df], axis=1)

print("Preprocesamiento manual terminado.")
print("Forma final del train:", X_train_manual.shape)
X_train_manual.head(3)

Preprocesamiento manual terminado.
Forma final del train: (16512, 13)


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,ocean_proximity_<1H OCEAN,ocean_proximity_INLAND,ocean_proximity_ISLAND,ocean_proximity_NEAR BAY,ocean_proximity_NEAR OCEAN
14196,1.272587,-1.372811,0.348490,0.222569,0.211228,0.768276,0.322906,-0.326196,0.0,0.0,0.0,0.0,1.0
8267,0.709162,-0.876696,1.618118,0.340293,0.593094,-0.098901,0.672027,-0.035843,0.0,0.0,0.0,0.0,1.0
17445,-0.447603,-0.460146,-1.952710,-0.342597,-0.495226,-0.449818,-0.430461,0.144701,0.0,0.0,0.0,0.0,1.0


In [17]:
# Entrenamos un modelo con los datos preprocesados a mano
modelo_manual = LinearRegression()
modelo_manual.fit(X_train_manual, y_train)

pred_manual = modelo_manual.predict(X_test_manual)
print("R² en prueba (flujo manual):", round(r2_score(y_test, pred_manual), 4))

R² en prueba (flujo manual): 0.6254


## 1.5. ¿Qué está mal con el flujo manual?


**1. Es frágil y propenso a la fuga de información.** Note cuántas veces tuvimos que recordar conscientemente usar `fit_transform` en el train y `transform` en el test. Un solo descuido —por ejemplo, escribir `scaler.fit_transform(X_test)`— introduce *data leakage* de forma **silenciosa**: el código no falla, no da error, pero el modelo queda contaminado y su evaluación deja de ser confiable. Este error es tristemente común.

**2. Es repetitivo y difícil de mantener.** Tuvimos que aplicar cada transformación **dos veces** (train y test), reconstruir DataFrames a mano y concatenar columnas. Si mañana llegan datos nuevos de producción, hay que repetir todo el ritual, en el mismo orden, sin olvidar ningún paso.

**3. No es transportable.** Para poner este modelo en producción tendríamos que serializar y cargar por separado el `scaler`, el `ohe`, la mediana, la lista de columnas... y volver a aplicarlos en el orden correcto. Cualquier desalineación rompe la predicción.

> La pregunta natural es: **¿y si pudiéramos empaquetar todos estos pasos en un único objeto que recuerde el orden, aprenda sólo del train y se aplique solo?** Esa es exactamente la promesa del **Pipeline**.

---
# Acto 2 — La solución: Pipeline y ColumnTransformer

## 2.1. ¿Qué es un Pipeline?

Un **`Pipeline`** es una **secuencia ordenada de pasos** que se comportan como un único objeto. Cada paso (excepto el último) es un **transformador** (tiene métodos `fit` y `transform`); el último suele ser un **estimador** (un modelo, con `fit` y `predict`).

La magia está en cómo se comporta:

- Cuando llamamos a **`pipeline.fit(X_train, y_train)`**, el pipeline aplica `fit_transform` a cada paso en cadena y finalmente entrena el modelo. **Aprende todos los parámetros sólo del entrenamiento.**
- Cuando llamamos a **`pipeline.predict(X_test)`**, aplica `transform` (¡no `fit`!) a cada paso con los parámetros ya aprendidos, y luego predice.

En otras palabras, **el pipeline hace por nosotros, y sin posibilidad de error, la disciplina que en el Acto 1 tuvimos que mantener manualmente**. Es imposible que se filtre información del test, porque el pipeline nunca hace `fit` sobre el test.

Empecemos con un pipeline sencillo, sólo para las variables numéricas.

In [18]:
# Un pipeline es una lista de pasos con nombre: [(nombre, transformador), ...]
pipeline_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # 1º rellena faltantes con la mediana
    ("scaler", StandardScaler())                     # 2º estandariza (media 0, desv. 1)
])

# Lo probamos sobre las columnas numéricas del entrenamiento.
resultado = pipeline_numerico.fit_transform(X_train[numericas])
print("El pipeline numérico transformó los datos. Forma:", resultado.shape)
print("Primeras filas ya imputadas y escaladas:")
print(np.round(resultado[:3], 3))

El pipeline numérico transformó los datos. Forma: (16512, 8)
Primeras filas ya imputadas y escaladas:
[[ 1.273 -1.373  0.348  0.223  0.211  0.768  0.323 -0.326]
 [ 0.709 -0.877  1.618  0.34   0.593 -0.099  0.672 -0.036]
 [-0.448 -0.46  -1.953 -0.343 -0.495 -0.45  -0.43   0.145]]


Ese pequeño pipeline ya encapsula **dos pasos** (imputar + escalar) en un solo objeto. Pero tenemos un problema pendiente: las columnas numéricas y la columna categórica necesitan **tratamientos distintos**. Un pipeline lineal aplica los mismos pasos a *todas* las columnas que recibe. Aquí entra el `ColumnTransformer`.

## 2.2. ¿Qué es el ColumnTransformer?

El **`ColumnTransformer`** permite aplicar **transformadores diferentes a subconjuntos de columnas diferentes**, en paralelo, y luego **une los resultados** en una sola matriz.

Es la pieza que resuelve el escenario más común del mundo real: *"a las columnas numéricas quiero escalarlas, pero a las categóricas quiero codificarlas con One-Hot"*. Visualmente:

```
                        ┌─────────────────────────────────────┐
   columnas numéricas → │ imputar (mediana) → escalar          │ ─┐
                        └─────────────────────────────────────┘  │
                                                                  ├─→ matriz final
                        ┌─────────────────────────────────────┐  │     (unida)
   columna categórica → │ One-Hot Encoding                     │ ─┘
                        └─────────────────────────────────────┘
```

Construyamos primero cada "rama" por separado y luego las combinamos.

In [19]:
# --- Rama NUMÉRICA: imputar + escalar ---
# (Añadimos PolynomialFeatures para generar términos cuadráticos, como en el material del curso.)
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scaler", StandardScaler())
])

# --- Rama CATEGÓRICA: One-Hot Encoding ---
# handle_unknown="ignore" evita que el modelo falle si en producción aparece
# una categoría nunca vista durante el entrenamiento.
transformador_categorico = Pipeline(steps=[
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

print("Ramas de transformación definidas: numérica y categórica.")

Ramas de transformación definidas: numérica y categórica.


In [20]:
# --- El ColumnTransformer une ambas ramas, asignando cada una a SUS columnas ---
preprocesador = ColumnTransformer(transformers=[
    ("num", transformador_numerico, numericas),      # rama numérica → columnas numéricas
    ("cat", transformador_categorico, categoricas)   # rama categórica → columna categórica
])

preprocesador

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('poly',
                                                  PolynomialFeatures(include_bias=False)),
                                                 ('scaler', StandardScaler())]),
                                 ['longitude', 'latitude', 'housing_median_age',
                                  'total_rooms', 'total_bedrooms', 'population',
                                  'households', 'median_income']),
                                ('cat',
                                 Pipeline(steps=[('onehot',
                                                  OneHotEncoder(handle_unknown='ignore'))]),
                                 ['ocean_proximity'])])

**Lo notable de esto:** el `ColumnTransformer` sabe qué transformación aplicar a cada columna **por su nombre**. Ya no tenemos que separar manualmente el DataFrame, transformar cada parte y volver a concatenar (todo lo tedioso del Acto 1). El `ColumnTransformer` lo hace internamente y **en el orden correcto**.

## 2.3. El pipeline completo: preprocesamiento + modelo

El paso final es elegante: metemos el `preprocesador` (que ya contiene ambas ramas) y el **modelo** dentro de **un único pipeline**. El resultado es un objeto que representa **todo el flujo de principio a fin**.

In [21]:
# Pipeline maestro: primero preprocesa, luego entrena el modelo.
modelo_completo = Pipeline(steps=[
    ("preprocesador", preprocesador),
    ("regresor", LinearRegression())
])

modelo_completo

Pipeline(steps=[('preprocesador',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('poly',
                                                                   PolynomialFeatures(include_bias=False)),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['longitude', 'latitude',
                                                   'housing_median_age',
                                                   'total_rooms',
                                                   'total_bedrooms',
                                                   'population', 'households',
                                                   'median_income']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['ocean_proximity'])])),
                ('regresor', LinearRegression())])

Observe el diagrama que scikit-learn dibuja arriba: es **todo nuestro flujo** representado como un solo objeto. Ahora viene la recompensa por todo el trabajo conceptual.

## 2.4. Entrenar y evaluar: todo en dos líneas

Compare esto con las decenas de líneas cuidadosas del Acto 1. Con el pipeline, entrenar es **una sola llamada a `fit`** y predecir es **una sola llamada a `predict`**. Internamente, el pipeline garantiza que:

- Los parámetros (mediana, media/desviación del scaler, categorías del One-Hot) se aprenden **sólo del train**.
- Al test se le aplican **exactamente las mismas** transformaciones con esos parámetros.

**La fuga de información se vuelve imposible por construcción.**

In [22]:
# Entrenamos TODO el flujo (imputación + polinomios + escalado + one-hot + modelo) de una vez.
modelo_completo.fit(X_train, y_train)

# Predecimos sobre el test crudo: el pipeline lo preprocesa solo, correctamente.
predicciones = modelo_completo.predict(X_test)

print("R² en prueba (pipeline):", round(r2_score(y_test, predicciones), 4))
print("Error absoluto medio:   ", round(mean_absolute_error(y_test, predicciones), 2), "USD")

R² en prueba (pipeline): 0.657
Error absoluto medio:    46981.22 USD


Fíjese en un detalle fundamental: a `predict` le pasamos **`X_test` crudo**, sin imputar, sin escalar, sin codificar. El pipeline se encargó de todo internamente. Eso es justamente lo que en el Acto 1 tuvimos que hacer manualmente y con riesgo de equivocarnos.

> **Este mismo objeto `modelo_completo`** puede recibir mañana un DataFrame de producción con la misma estructura y devolver predicciones correctas, sin ningún paso adicional. Es transportable y reproducible.

---

Hasta aquí el pipeline da código limpio y seguridad frente al *data leakage*. Pero su verdadero potencial aparece cuando lo combinamos con las demás herramientas de scikit-learn, que están **diseñadas para trabajar con pipelines**.

## 3.1. Validación cruzada correcta, gratis

La **validación cruzada** divide los datos en varios pliegues y evalúa el modelo repetidamente para obtener una estimación más robusta de su desempeño. El problema es que, hecha a mano, es **muy fácil filtrar información**: si escalamos antes de dividir en pliegues, cada pliegue de validación "ve" información de los demás.

Con un pipeline esto se resuelve solo: al pasar el pipeline completo a `cross_val_score`, **el preprocesamiento se re-aprende dentro de cada pliegue**, usando sólo la porción de entrenamiento de ese pliegue. Correcto por construcción.

In [23]:
# Validación cruzada de 5 pliegues sobre TODO el pipeline.
# En cada pliegue, el preprocesador se ajusta sólo con los datos de entrenamiento de ese pliegue.
scores = cross_val_score(modelo_completo, X_train, y_train, cv=5, scoring="r2")

print("R² en cada uno de los 5 pliegues:", np.round(scores, 4))
print("R² promedio:", round(scores.mean(), 4), "±", round(scores.std(), 4))

R² en cada uno de los 5 pliegues: [0.7017 0.7005 0.7062 0.7073 0.3536]
R² promedio: 0.6339 ± 0.1401


## 3.2. Búsqueda de hiperparámetros sobre TODO el flujo

Aquí se aprecia por qué el pipeline es tan poderoso. En el material original del curso, para probar `PolynomialFeatures` con `degree=2` y luego `degree=3`, había que **reescribir y re-ejecutar varias celdas a mano**. Con un pipeline, podemos pedirle a `GridSearchCV` que pruebe automáticamente varias configuraciones y elija la mejor por validación cruzada.

La sintaxis para acceder a un parámetro interno es `nombre_del_paso__parametro`, usando **doble guion bajo** (`__`). Como el paso `poly` está dentro de la rama `num`, dentro del `preprocesador`, la ruta completa es:

`preprocesador__num__poly__degree`

In [24]:
# Definimos la grilla de valores a probar para el grado del polinomio.
param_grid = {
    "preprocesador__num__poly__degree": [1, 2, 3]
}

# GridSearchCV probará cada grado con validación cruzada y se quedará con el mejor.
busqueda = GridSearchCV(
    modelo_completo,
    param_grid,
    cv=3,
    scoring="r2",
    n_jobs=-1
)

busqueda.fit(X_train, y_train)

print("Mejor grado de PolynomialFeatures:", busqueda.best_params_)
print("Mejor R² (validación cruzada):    ", round(busqueda.best_score_, 4))

Mejor grado de PolynomialFeatures: {'preprocesador__num__poly__degree': 1}
Mejor R² (validación cruzada):     0.6478


In [25]:
# Evaluamos el MEJOR pipeline encontrado sobre el conjunto de prueba.
mejor_modelo = busqueda.best_estimator_
pred_mejor = mejor_modelo.predict(X_test)

print("R² en prueba con el mejor pipeline:", round(r2_score(y_test, pred_mejor), 4))

R² en prueba con el mejor pipeline: 0.6254


Con **una sola búsqueda** exploramos tres configuraciones distintas de preprocesamiento y modelo, cada una evaluada correctamente por validación cruzada, y obtuvimos el mejor flujo completo ya entrenado. Hacer esto a mano habría requerido decenas de celdas y una enorme disciplina para no equivocarse.

## 3.3. Inspeccionar el pipeline

Un pipeline no es una caja negra: podemos asomarnos a cualquiera de sus pasos. Por ejemplo, veamos cuántas columnas generó el preprocesamiento (los términos polinómicos y las categorías One-Hot expanden bastante el número de features).

In [26]:
# Accedemos al preprocesador ya entrenado dentro del mejor pipeline.
prep_entrenado = mejor_modelo.named_steps["preprocesador"]

# Número de columnas que entran vs. columnas que salen del preprocesamiento.
n_salida = prep_entrenado.transform(X_train).shape[1]
print("Columnas originales de entrada:", X_train.shape[1])
print("Columnas tras el preprocesamiento:", n_salida)
print("\nEl ColumnTransformer expandió las features combinando polinomios y One-Hot.")

Columnas originales de entrada: 9
Columnas tras el preprocesamiento: 13

El ColumnTransformer expandió las features combinando polinomios y One-Hot.


---
# Resumen y conclusiones

Recorrimos el camino completo, desde el problema hasta el potencial:

| Aspecto | Flujo manual (Acto 1) | Con Pipeline + ColumnTransformer |
|---|---|---|
| **Riesgo de *data leakage*** | Alto: depende de la disciplina del analista | **Nulo por construcción**: nunca se hace `fit` sobre el test |
| **Cantidad de código** | Cada paso repetido para train y test | `fit` una vez, `predict` una vez |
| **Columnas de distinto tipo** | Separar, transformar y concatenar a mano | El `ColumnTransformer` lo hace automáticamente |
| **Validación cruzada correcta** | Muy difícil de hacer bien a mano | `cross_val_score(pipeline, ...)` y listo |
| **Ajuste de hiperparámetros** | Reescribir celdas manualmente | `GridSearchCV` sobre todo el flujo |
| **Puesta en producción** | Serializar cada objeto por separado | Un único objeto transportable |


1. **Un Pipeline encadena pasos y aplica la disciplina `fit`-en-train / `transform`-en-test de forma automática.** Esto elimina la principal fuente de fuga de información.

2. **El ColumnTransformer aplica transformaciones distintas a columnas distintas** (numéricas → escalar; categóricas → One-Hot) y une los resultados. Es la pieza clave para datasets del mundo real, que casi siempre mezclan tipos.

3. **Un pipeline no es sólo comodidad, es corrección.** Al integrarse con `cross_val_score` y `GridSearchCV`, permite evaluar y optimizar **todo el flujo** —preprocesamiento incluido— sin filtrar información y sin reescribir código.

> En la práctica profesional, **casi todo modelo de machine learning debería vivir dentro de un pipeline**. No es una técnica avanzada opcional: es la forma correcta y estándar de trabajar.

---

*Reto: reemplazar `LinearRegression` por otro estimador (por ejemplo `RandomForestRegressor`) cambiando **una sola línea** del pipeline, y comprobar que todo el resto del flujo sigue funcionando sin modificaciones. Ese es, precisamente, el poder de los pipelines.*